In [1]:
!pip install pandas numpy

In [2]:
import re
import pandas as pd
from datetime import datetime

In [3]:
months_ua = {
    "січня": "01",
    "лютого": "02",
    "березня": "03",
    "квітня": "04",
    "травня": "05",
    "червня": "06",
    "липня": "07",
    "серпня": "08",
    "вересня": "09",
    "жовтня": "10",
    "листопада": "11",
    "грудня": "12"
}

currencies = {
    "грн": "UAH",
    "₴": "UAH",
    "uah": "UAH",
    "$": "USD",
    "usd": "USD",
    "€": "EUR",
    "eur": "EUR"
}

In [4]:
date_regex = r'\b(\d{1,2})[./](\d{1,2})[./](\d{4})\b'
amount_regex = r'\b(\d+[.,]?\d*)\s?(грн|₴|uah|usd|\$|eur|€)\b'
phone_regex = r'\+?380\d{9}|\b0\d{9}\b'

In [5]:
def extract_dates(text):

    results = []

    for m in re.finditer(date_regex, text):

        day, month, year = m.groups()

        try:
            parsed = datetime(int(year), int(month), int(day))
            norm = parsed.strftime("%Y-%m-%d")
        except:
            norm = None

        results.append({
            "field_type": "DATE",
            "value": norm,
            "raw": m.group(),
            "start_char": m.start(),
            "end_char": m.end(),
            "method": "regex_date_v1"
        })

    return results

In [6]:
def extract_amounts(text):

    results = []

    for m in re.finditer(amount_regex, text.lower()):

        value, cur = m.groups()

        value = float(value.replace(",", "."))

        currency = currencies.get(cur, "UNKNOWN")

        results.append({
            "field_type": "AMOUNT",
            "value": value,
            "currency": currency,
            "start_char": m.start(),
            "end_char": m.end(),
            "method": "regex_amount_v1"
        })

    return results

In [7]:
def normalize_phone(phone):

    phone = phone.replace(" ", "").replace("-", "")

    if phone.startswith("0"):
        phone = "+38" + phone

    if not phone.startswith("+"):
        phone = "+" + phone

    return phone

In [8]:
def extract_phones(text):

    results = []

    for m in re.finditer(phone_regex, text):

        raw = m.group()

        results.append({
            "field_type": "PHONE",
            "value": normalize_phone(raw),
            "raw": raw,
            "start_char": m.start(),
            "end_char": m.end(),
            "method": "regex_phone_v1"
        })

    return results

In [9]:
def extract_all(text):

    results = []

    results += extract_dates(text)
    results += extract_amounts(text)
    results += extract_phones(text)

    return results

In [10]:
texts = [
    "Договір укладено 12.05.2024 на суму 5000 грн.",
    "Телефон постачальника +380671234567.",
    "Оплата 100 USD здійснена 01.03.2023",
    "Номер для зв'язку 0671234567"
]

df = pd.DataFrame({"text":texts})
df

,text
0,Договір укладено 12.05.2024 на суму 5000 грн.
1,Телефон постачальника +380671234567.
2,Оплата 100 USD здійснена 01.03.2023
3,Номер для зв'язку 0671234567


In [11]:
df["extractions"] = df["text"].apply(extract_all)

df

,text,extractions
0,Договір укладено 12.05.2024 на суму 5000 грн.,"[{'field_type': 'DATE', 'value': '2024-05-12',..."
1,Телефон постачальника +380671234567.,"[{'field_type': 'PHONE', 'value': '+3806712345..."
2,Оплата 100 USD здійснена 01.03.2023,"[{'field_type': 'DATE', 'value': '2023-03-01',..."
3,Номер для зв'язку 0671234567,"[{'field_type': 'PHONE', 'value': '+3806712345..."


In [12]:
gold = [
    {"text":"Договір укладено 12.05.2024 на суму 5000 грн", "field_type":"DATE"},
    {"text":"Телефон +380671234567", "field_type":"PHONE"},
    {"text":"Оплата 100 USD", "field_type":"AMOUNT"},
]

gold_df = pd.DataFrame(gold)
gold_df

,text,field_type
0,Договір укладено 12.05.2024 на суму 5000 грн,DATE
1,Телефон +380671234567,PHONE
2,Оплата 100 USD,AMOUNT


In [13]:
def evaluate_precision(texts):

    correct = 0
    total = 0

    for t in texts:

        preds = extract_all(t)

        total += len(preds)

        if len(preds) > 0:
            correct += 1

    precision = correct / total if total > 0 else 0

    return precision

In [14]:
precision = evaluate_precision(texts)

print("Precision:", precision)

Precision: 0.6666666666666666


In [15]:
for t in texts:

    preds = extract_all(t)

    print("TEXT:", t)
    print(preds)
    print()

TEXT: Договір укладено 12.05.2024 на суму 5000 грн.
[{'field_type': 'DATE', 'value': '2024-05-12', 'raw': '12.05.2024', 'start_char': 17, 'end_char': 27, 'method': 'regex_date_v1'}, {'field_type': 'AMOUNT', 'value': 5000.0, 'currency': 'UAH', 'start_char': 36, 'end_char': 44, 'method': 'regex_amount_v1'}]

TEXT: Телефон постачальника +380671234567.
[{'field_type': 'PHONE', 'value': '+380671234567', 'raw': '+380671234567', 'start_char': 22, 'end_char': 35, 'method': 'regex_phone_v1'}]

TEXT: Оплата 100 USD здійснена 01.03.2023
[{'field_type': 'DATE', 'value': '2023-03-01', 'raw': '01.03.2023', 'start_char': 25, 'end_char': 35, 'method': 'regex_date_v1'}, {'field_type': 'AMOUNT', 'value': 100.0, 'currency': 'USD', 'start_char': 7, 'end_char': 14, 'method': 'regex_amount_v1'}]

TEXT: Номер для зв'язку 0671234567
[{'field_type': 'PHONE', 'value': '+380671234567', 'raw': '0671234567', 'start_char': 18, 'end_char': 28, 'method': 'regex_phone_v1'}]



In [16]:
edge_cases = [

"100%",
"2024",
"номер 123",
"ціна 10%",
"тел 067123"
]

for e in edge_cases:

    print(e, extract_all(e))

100% []
2024 []
номер 123 []
ціна 10% []
тел 067123 []
